# Validación de audit logs y logrotate en Kubernetes

Este notebook comprueba el almacenamiento persistente de auditoría del clúster Vault PR, los montajes de los pods, el audit device de Vault y el sidecar de `logrotate`.

Las comprobaciones normales son de solo lectura. La última prueba fuerza una rotación únicamente si se cambia `FORCE_ROTATION=0` por `FORCE_ROTATION=1`.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if ENV_FILE:
    load_dotenv(ENV_FILE, override=True)

os.environ.setdefault("KUBE_CONTEXT", "PR")
os.environ.setdefault("VAULT_K8S_NAMESPACE", "vaultpr")
os.environ.setdefault("VAULT_HELM_RELEASE_NAME", "vaultpr")
os.environ.setdefault("VAULT_AUDIT_PATH", "/vault/audit/vault.log")

print(f"Contexto:  {os.environ['KUBE_CONTEXT']}")
print(f"Namespace: {os.environ['VAULT_K8S_NAMESPACE']}")
print(f"Release:   {os.environ['VAULT_HELM_RELEASE_NAME']}")
print(f".env:      {ENV_FILE or 'no encontrado (solo será necesario para consultar Vault)'}")

Contexto:  PR
Namespace: vaultpr
Release:   vaultpr
.env:      /Users/jose/Library/CloudStorage/GoogleDrive-jose.maria.merchan@gmail.com/My Drive/Demo/Mapfre_PoC_Vault/.env


## 1. Contexto, StatefulSet y pods

Verifica que el contexto existe, que el StatefulSet está disponible y que cada pod incluye los contenedores `vault` y `auditlog-rotator`.

In [2]:
%%bash
set -euo pipefail

kubectl config get-contexts "${KUBE_CONTEXT}"
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o wide
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' \
  -o custom-columns='POD:.metadata.name,READY:.status.containerStatuses[*].ready,CONTAINERS:.spec.containers[*].name,RESTARTS:.status.containerStatuses[*].restartCount'

not_ready=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o json | \
  jq '[.items[].status.containerStatuses[] | select(.ready != true)] | length')
test "${not_ready}" -eq 0
echo "OK: todos los contenedores de Vault están Ready."

CURRENT   NAME   CLUSTER   AUTHINFO   NAMESPACE
          PR     PR        PR         default
NAME      READY   AGE   CONTAINERS               IMAGES
vaultpr   3/3     15m   vault,auditlog-rotator   hashicorp/vault-enterprise:2.0.3-ent,josemerchan/vault-logrotate:0.0.1
POD         READY       CONTAINERS               RESTARTS
vaultpr-0   true,true   vault,auditlog-rotator   0,0
vaultpr-1   true,true   vault,auditlog-rotator   0,0
vaultpr-2   true,true   vault,auditlog-rotator   0,0
OK: todos los contenedores de Vault están Ready.


## 2. PVC de auditoría

Comprueba que existe un PVC de auditoría por réplica, que todos están en estado `Bound` y muestra capacidad, StorageClass y volumen asociado.

In [3]:
%%bash
set -euo pipefail

replicas=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o jsonpath='{.spec.replicas}')
audit_pvcs=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pvc -o json | jq --arg release "${VAULT_HELM_RELEASE_NAME}" \
  '[.items[] | select(.metadata.name | startswith("audit-" + $release + "-"))]')

echo "${audit_pvcs}" | jq -r \
  '["PVC","STATUS","CAPACITY","STORAGE_CLASS","VOLUME"],
   (.[] | [.metadata.name,.status.phase,.status.capacity.storage,.spec.storageClassName,.spec.volumeName]) | @tsv' | column -t

count=$(jq 'length' <<<"${audit_pvcs}")
bound=$(jq '[.[] | select(.status.phase == "Bound")] | length' <<<"${audit_pvcs}")
test "${count}" -eq "${replicas}"
test "${bound}" -eq "${replicas}"
echo "OK: ${bound}/${replicas} PVC de auditoría están Bound."

PVC              STATUS  CAPACITY  STORAGE_CLASS  VOLUME
audit-vaultpr-0  Bound   1Gi       standard       pvc-8d412982-c493-4472-80a3-139b8402d50b
audit-vaultpr-1  Bound   1Gi       standard       pvc-1b7e2f3f-c9cc-4daa-ad6a-230c40e1e6a9
audit-vaultpr-2  Bound   1Gi       standard       pvc-1322b223-02e7-4a93-877c-ab6e73e41614
OK: 3/3 PVC de auditoría están Bound.


## 3. Volúmenes, montajes y process namespace

Valida que Vault y el sidecar comparten `/vault/audit`, que el ConfigMap se monta como `/etc/logrotate.conf` y que `shareProcessNamespace` permite al sidecar enviar `SIGHUP` a Vault.

In [4]:
%%bash
set -euo pipefail

sts=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o json)

jq -r '.spec.template.spec.containers[] as $c |
  $c.volumeMounts[]? |
  select(.mountPath == "/vault/audit" or .mountPath == "/etc/logrotate.conf") |
  [$c.name,.name,.mountPath,(.subPath // "-")] | @tsv' <<<"${sts}" | \
  { printf 'CONTAINER\tVOLUME\tMOUNT\tSUBPATH\n'; cat; } | column -t

jq -e '.spec.template.spec.shareProcessNamespace == true' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "vault") |
  .volumeMounts[] | select(.name == "audit" and .mountPath == "/vault/audit")] | length == 1' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "auditlog-rotator") |
  .volumeMounts[] | select(.name == "audit" and .mountPath == "/vault/audit")] | length == 1' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "auditlog-rotator") |
  .volumeMounts[] | select(.name == "logrotate-config" and .mountPath == "/etc/logrotate.conf")] | length == 1' <<<"${sts}" >/dev/null
echo "OK: montajes y process namespace configurados correctamente."

CONTAINER         VOLUME            MOUNT                SUBPATH
vault             audit             /vault/audit         -
auditlog-rotator  logrotate-config  /etc/logrotate.conf  logrotate.conf
auditlog-rotator  audit             /vault/audit         -
OK: montajes y process namespace configurados correctamente.


## 4. Configuración y procesos de logrotate

Muestra la política efectiva, la planificación del sidecar, sus procesos y sus últimos logs. También ejecuta `logrotate` en modo debug, que no modifica los ficheros.

In [5]:
%%bash
set -euo pipefail

pod="${VAULT_HELM_RELEASE_NAME}-0"
echo '--- /etc/logrotate.conf ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- cat /etc/logrotate.conf
echo '--- CRONTAB y procesos ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- sh -c 'echo "CRONTAB=${CRONTAB:-unset}"; ps'
echo '--- logrotate debug (sin cambios) ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- logrotate -d /etc/logrotate.conf
echo '--- logs del sidecar ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  logs "${pod}" -c auditlog-rotator --tail=50
echo 'OK: configuración de logrotate legible y válida.'

--- /etc/logrotate.conf ---
/vault/audit/vault.log {
    rotate 2
    size 1M
    missingok
    notifempty
    compress

    postrotate
        pkill -HUP -x vault
    endscript
}
--- CRONTAB y procesos ---
CRONTAB=*/5 * * * *
PID   USER     TIME  COMMAND
    1 crond     0:00 /pause
    7 crond     0:00 /bin/sh -ec cp /vault/config/extraconfig-from-values.hcl /tmp/storageconfig.hcl; [ -n "${HOST_IP}" ] && sed -Ei "s|HOST_IP|${HOST_IP?}|g" /tmp/storageconfig.hcl; [ -n "${POD_IP}" ] && sed -Ei "s|POD_IP|${POD_IP?}|g" /tmp/storageconfig.hcl; [ -n "${HOSTNAME}" ] && sed -Ei "s|HOSTNAME|${HOSTNAME?}|g" /tmp/storageconfig.hcl; [ -n "${API_ADDR}" ] && sed -Ei "s|API_ADDR|${API_ADDR?}|g" /tmp/storageconfig.hcl; [ -n "${TRANSIT_ADDR}" ] && sed -Ei "s|TRANSIT_ADDR|${TRANSIT_ADDR?}|g" /tmp/storageconfig.hcl; [ -n "${RAFT_ADDR}" ] && sed -Ei "s|RAFT_ADDR|${RAFT_ADDR?}|g" /tmp/storageconfig.hcl; if grep -vE '^[[:space:]]*(#|//)' /tmp/storageconfig.hcl | grep -qE '\"?autopilot_redundancy_zone\"?[[:s


reading config file /etc/logrotate.conf
Reading state from file: /var/lib/logrotate.status
state file /var/lib/logrotate.status does not exist
Allocating hash table for state file, size 64 entries

Handling 1 logs

rotating pattern: /vault/audit/vault.log  1048576 bytes (2 rotations)
empty log files are not rotated, old logs are removed
considering log /vault/audit/vault.log
Creating new state
  Now: 2026-07-27 16:55
  Last rotated at 2026-07-27 16:00
  log does not need rotating (log size is below the 'size' threshold)


--- logs del sidecar ---
Starting vault logrotation with "*/5 * * * *"
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
OK: configuración de logrotate legible y válida.


## 5. Audit device de Vault

Consulta `sys/audit` desde `vaultpr-0` y confirma que existe un audit device de tipo `file` apuntando a `/vault/audit/vault.log`. La autenticación utiliza el usuario `tester` y requiere `VAULT_PR_TESTER_PASSWORD` en el fichero `.env`; el token secundario solo vive durante la celda.

In [6]:
%%bash
set -euo pipefail
: "${VAULT_PR_TESTER_PASSWORD:?VAULT_PR_TESTER_PASSWORD no está definido; añádelo al fichero .env}"

pod="${VAULT_HELM_RELEASE_NAME}-0"
tester_login=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true \
  vault login -format=json -method=userpass username=tester password="${VAULT_PR_TESTER_PASSWORD}")
secondary_token=$(jq -r '.auth.client_token' <<<"${tester_login}")
unset tester_login
audit_devices=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true VAULT_TOKEN="${secondary_token}" \
  vault audit list -format=json)
unset secondary_token

jq -r 'to_entries[] | [.key,.value.type,(.value.options.file_path // "-")] | @tsv' \
  <<<"${audit_devices}" | { printf 'PATH\tTYPE\tFILE\n'; cat; } | column -t
jq -e --arg path "${VAULT_AUDIT_PATH}" \
  'to_entries | any(.value.type == "file" and .value.options.file_path == $path)' \
  <<<"${audit_devices}" >/dev/null
echo "OK: audit device file activo en ${VAULT_AUDIT_PATH}."

PATH   TYPE  FILE
file/  file  /vault/audit/vault.log
OK: audit device file activo en /vault/audit/vault.log.


## 6. Ficheros de auditoría en todas las réplicas

Comprueba existencia, permisos, tamaño y sintaxis JSON. Solo muestra campos operativos de las últimas entradas para evitar volcar el registro de auditoría completo.

In [7]:
%%bash
set -euo pipefail

pods=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o jsonpath='{.items[*].metadata.name}')
for pod in ${pods}; do
  echo "--- ${pod} ---"
  kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- sh -ec 'test -s /vault/audit/vault.log; ls -lh /vault/audit/vault.log*'
  recent=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- tail -n 20 /vault/audit/vault.log)
  jq -e . >/dev/null <<<"${recent}"
  tail -n 3 <<<"${recent}" | \
    jq -r '[.time,.type,(.request.operation // "-"),(.request.path // "-")] | @tsv'
done
echo 'OK: los audit logs existen y las últimas entradas contienen JSON válido.'

--- vaultpr-0 ---
-rw-------    1 vault    vault      15.3K Jul 27 16:55 /vault/audit/vault.log
-rw-------    1 vault    vault       4.3K Jul 27 16:41 /vault/audit/vault.log.1.gz
2026-07-27T16:55:12.141052422Z	response	update	auth/userpass/login/tester
2026-07-27T16:55:55.382859386Z	request	update	auth/userpass/login/tester
2026-07-27T16:55:55.541541928Z	response	update	auth/userpass/login/tester
--- vaultpr-1 ---
-rw-------    1 vault    vault      13.5K Jul 27 16:55 /vault/audit/vault.log
2026-07-27T16:55:12.25414688Z	response	read	sys/audit
2026-07-27T16:55:55.655797345Z	request	read	sys/audit
2026-07-27T16:55:55.655895136Z	response	read	sys/audit
--- vaultpr-2 ---
-rw-------    1 vault    vault       4.7K Jul 27 16:41 /vault/audit/vault.log
2026-07-27T16:41:17.501708841Z	response	update	sys/audit/file
2026-07-27T16:41:19.690470009Z	request	read	sys/audit
2026-07-27T16:41:19.690554967Z	response	read	sys/audit
OK: los audit logs existen y las últimas entradas contienen JSON válido.


## 7. Persistencia asociada al pod

Relaciona cada pod con su PVC de auditoría y muestra el uso real del filesystem desde el contenedor Vault.

In [8]:
%%bash
set -euo pipefail

for pod in $(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o name | cut -d/ -f2); do
  claim=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" get pod "${pod}" \
    -o json | jq -r '.spec.volumes[] | select(.name == "audit") | .persistentVolumeClaim.claimName')
  printf '%s -> %s\n' "${pod}" "${claim}"
  kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- df -h /vault/audit
done

vaultpr-0 -> audit-vaultpr-0
Filesystem                Size      Used Available Use% Mounted on
/dev/vda1                 1.8T     15.2G      1.7T   1% /vault/file
vaultpr-1 -> audit-vaultpr-1
Filesystem                Size      Used Available Use% Mounted on
/dev/vda1                 1.8T     15.2G      1.7T   1% /etc/hostname
vaultpr-2 -> audit-vaultpr-2
Filesystem                Size      Used Available Use% Mounted on
/dev/vda1                 1.8T     15.2G      1.7T   1% /vault/logs


## 8. Prueba funcional opcional de rotación

Al habilitarla, fuerza una rotación en `vaultpr-0`, genera una operación auditada y confirma que existen tanto el fichero rotado como el nuevo fichero activo. Esta prueba modifica únicamente los ficheros de auditoría conforme a la política instalada.

In [10]:
%%bash
set -euo pipefail

FORCE_ROTATION=1
if [[ "${FORCE_ROTATION}" != "1" ]]; then
  echo 'Prueba omitida. Cambia FORCE_ROTATION=1 para ejecutarla.'
  exit 0
fi
: "${VAULT_PR_TESTER_PASSWORD:?VAULT_PR_TESTER_PASSWORD no está definido; añádelo al fichero .env}"

pod="${VAULT_HELM_RELEASE_NAME}-0"
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- \
  logrotate -s /tmp/logrotate-force.status -f /etc/logrotate.conf
tester_login=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true \
  vault login -format=json -method=userpass username=tester password="${VAULT_PR_TESTER_PASSWORD}")
secondary_token=$(jq -r '.auth.client_token' <<<"${tester_login}")
unset tester_login
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true VAULT_TOKEN="${secondary_token}" \
  vault token lookup -format=json >/dev/null
unset secondary_token

kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- sh -ec '
    test -s /vault/audit/vault.log
    find /vault/audit -maxdepth 1 -type f -name "vault.log.*" | grep -q .
    ls -lh /vault/audit/vault.log*
  '
last_entry=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c vault -- tail -n 1 /vault/audit/vault.log)
jq -e . >/dev/null <<<"${last_entry}"
echo 'OK: rotación forzada, reapertura mediante SIGHUP y nueva escritura validadas.'

error: error creating stub state file /var/lib/logrotate.status: Permission denied
command terminated with exit code 3


CalledProcessError: Command 'b'set -euo pipefail\n\nFORCE_ROTATION=1\nif [[ "${FORCE_ROTATION}" != "1" ]]; then\n  echo \'Prueba omitida. Cambia FORCE_ROTATION=1 para ejecutarla.\'\n  exit 0\nfi\n: "${VAULT_PR_TESTER_PASSWORD:?VAULT_PR_TESTER_PASSWORD no est\xc3\xa1 definido; a\xc3\xb1\xc3\xa1delo al fichero .env}"\n\npod="${VAULT_HELM_RELEASE_NAME}-0"\nkubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \\\n  exec "${pod}" -c auditlog-rotator -- logrotate -f /etc/logrotate.conf\ntester_login=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \\\n  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true \\\n  vault login -format=json -method=userpass username=tester password="${VAULT_PR_TESTER_PASSWORD}")\nsecondary_token=$(jq -r \'.auth.client_token\' <<<"${tester_login}")\nunset tester_login\nkubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \\\n  exec "${pod}" -c vault -- env VAULT_SKIP_VERIFY=true VAULT_TOKEN="${secondary_token}" \\\n  vault to